In [ ]:
from pathlib import Path
import sys

import torch
from PIL import Image
from transformers import CLIPTokenizer

import model_loader
from collections import defaultdict

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "sd" else Path.cwd()
SD_DIR = ROOT_DIR / "sd"
DATA_DIR = ROOT_DIR / "data"
OUTPUT_PATH = ROOT_DIR / "images//output.png"
BASE_CKPT_PATH = DATA_DIR / "v1-5-pruned-emaonly.ckpt"
LORA_CKPT_PATH = DATA_DIR / "xinminzu.safetensors"


if str(SD_DIR) not in sys.path:
    sys.path.insert(0, str(SD_DIR))

import pipeline


def main() -> None:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    if not BASE_CKPT_PATH.exists():
        raise FileNotFoundError(f"Base checkpoint not found: {BASE_CKPT_PATH}")
    if not LORA_CKPT_PATH.exists():
        raise FileNotFoundError(f"LoRA checkpoint not found: {LORA_CKPT_PATH}")

    tokenizer = CLIPTokenizer(
        vocab_file=str(DATA_DIR / "vocab.json"),
        merges_file=str(DATA_DIR / "merges.txt"),
    )

    models = model_loader.preload_models_from_standard_weights(str(BASE_CKPT_PATH), device)

    # quick debug wrapper
    try:
        info = model_loader.load_lora_weights_into_unet(
            models["diffusion"].unet,
            checkpoint_path=str(LORA_CKPT_PATH),
            device=device,
            default_alpha=1.0,
        )
        print("LoRA loaded:", info)
    except Exception as e:
        import traceback
        traceback.print_exc()
        # inspect raw state dict keys to find malformed prefix
        from pathlib import Path
        p = Path(LORA_CKPT_PATH)
        from model_loader import _load_state_dict_from_checkpoint, _normalize_lora_key
        sd = _load_state_dict_from_checkpoint(str(p), device)
        print("Found candidate LoRA keys (normalized):")
        for k in sd.keys():
            nk = _normalize_lora_key(k)
            if nk.endswith(("q_proj","k_proj","v_proj")) or "lora_up" in nk or "lora_down" in nk:
                print(k, "->", nk)
        raise

    prompt = "Chinese 1950s-1980s propaganda poster, revolutionary poster style, agricultural harvest,farmers, golden rice fields, wheat waves, full granaries, tractors, farmland water conservancy,farmers with joyful smiles, simple and optimistic expressions"
    negative_prompt = "blurry, low quality, distorted, bad anatomy, artifacts"
    cfg_scale = 8.0
    seed = 42

    image_array = pipeline.generate(
        prompt=prompt,
        uncond_prompt=negative_prompt,
        input_image=None,
        strength=0.8,
        do_cfg=True,
        cfg_scale=cfg_scale,
        sampler_name="ddpm",
        n_inference_steps=50,
        models=models,
        seed=seed,
        device=device,
        idle_device="cpu",
        tokenizer=tokenizer,
    )
    Image.fromarray(image_array)
    Image.fromarray(image_array).save(OUTPUT_PATH)
    print(f"Saved image to {OUTPUT_PATH}")


main()

c:\SD\pytorch-stable-diffusion-UI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


c:\SD\pytorch-stable-diffusion-UI\.venv\Lib\site-packages\lightning_fabric\__init__.py:41: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


LoRA loaded: {'loaded_modules': [], 'skipped_prefixes': ['lora_te_text_model_encoder_layers_0_mlp_fc1', 'lora_te_text_model_encoder_layers_0_mlp_fc2', 'lora_te_text_model_encoder_layers_0_self_attn_k_proj', 'lora_te_text_model_encoder_layers_0_self_attn_out_proj', 'lora_te_text_model_encoder_layers_0_self_attn_q_proj', 'lora_te_text_model_encoder_layers_0_self_attn_v_proj', 'lora_te_text_model_encoder_layers_10_mlp_fc1', 'lora_te_text_model_encoder_layers_10_mlp_fc2', 'lora_te_text_model_encoder_layers_10_self_attn_k_proj', 'lora_te_text_model_encoder_layers_10_self_attn_out_proj', 'lora_te_text_model_encoder_layers_10_self_attn_q_proj', 'lora_te_text_model_encoder_layers_10_self_attn_v_proj', 'lora_te_text_model_encoder_layers_11_mlp_fc1', 'lora_te_text_model_encoder_layers_11_mlp_fc2', 'lora_te_text_model_encoder_layers_11_self_attn_k_proj', 'lora_te_text_model_encoder_layers_11_self_attn_out_proj', 'lora_te_text_model_encoder_layers_11_self_attn_q_proj', 'lora_te_text_model_encoder_

100%|██████████| 50/50 [25:51<00:00, 31.02s/it]


Saved image to c:\SD\pytorch-stable-diffusion-UI\images\output.png


In [2]:
import torch
print(torch.cuda.is_available())  # 输出True即为成功
print(torch.version.cuda)         # 输出13.0，确认CUDA版本匹配

True
11.8
